In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Echo times (61 points -> rapide et fluide)
TE = np.arange(0, 305, 5)

# Grille de temps T2 pour le spectre (ms)
T2_grid = np.linspace(5, 180, 200)

def gaussian_peak(x, mu, sigma, area):
    g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)
    return area * g / np.trapezoid(g, x)

# Paramètres du tissu sain de référence
HEALTHY_MWF = 20
T2_MW = 20
T2_IEW_H = 80
T2_IEW_D = 105  # Légère hausse du T2 IEW (œdème/inflammation en lésion)

# Courbe et spectre du tissu sain
S_healthy = (HEALTHY_MWF / 100) * np.exp(-TE / T2_MW) + (1 - HEALTHY_MWF / 100) * np.exp(-TE / T2_IEW_H)
spec_healthy = (
    gaussian_peak(T2_grid, T2_MW, 3.0, HEALTHY_MWF / 100)
    + gaussian_peak(T2_grid, T2_IEW_H, 8.0, 1 - HEALTHY_MWF / 100)
)

# Valeurs de MWF pour le tissu démyélinisé
demyelinated_values = [2, 5, 8, 10, 12, 15, 20]
initial_dem = 5

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Biexponential T2 Signal Decay", "Corresponding T2 Distribution (NNLS Spectrum)")
)

# Traces fixes : Tissu sain (indices 0 et 1)
fig.add_trace(
    go.Scatter(
        x=TE, y=S_healthy,
        name=f"Healthy WM (MWF = {HEALTHY_MWF}%)",
        line=dict(color="#1f77b4", width=3),
        visible=True
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=T2_grid, y=spec_healthy,
        name=f"Healthy spectrum",
        line=dict(color="#1f77b4", width=2.5),
        fill="tozeroy",
        fillcolor="rgba(31, 119, 180, 0.15)",
        showlegend=False,
        visible=True
    ),
    row=1, col=2
)

# Traces dynamiques : Tissu démyélinisé
for d_mwf in demyelinated_values:
    vis = (d_mwf == initial_dem)
    S_dem = (d_mwf / 100) * np.exp(-TE / T2_MW) + (1 - d_mwf / 100) * np.exp(-TE / T2_IEW_D)
    spec_dem = (
        gaussian_peak(T2_grid, T2_MW, 3.0, d_mwf / 100)
        + gaussian_peak(T2_grid, T2_IEW_D, 10.0, 1 - d_mwf / 100)
    )

    fig.add_trace(
        go.Scatter(
            x=TE, y=S_dem,
            name="Demyelinated tissue",
            line=dict(color="#d62728", width=3, dash="dash"),
            visible=vis
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=T2_grid, y=spec_dem,
            name="Demyelinated spectrum",
            line=dict(color="#d62728", width=2.5, dash="dash"),
            fill="tozeroy",
            fillcolor="rgba(214, 39, 40, 0.15)",
            showlegend=False,
            visible=vis
        ),
        row=1, col=2
    )

# Ligne de coupure MWF (T2 = 40 ms) sur le spectre
fig.add_vline(
    x=40, line_dash="dot", line_color="gray",
    annotation_text="MWF cutoff (40 ms)", annotation_position="top right",
    row=1, col=2
)

# Étapes du curseur
total_traces = len(fig.data)
steps = []
for idx, d_mwf in enumerate(demyelinated_values):
    visibility = [True, True] + [False] * (2 * len(demyelinated_values))
    visibility[2 + 2 * idx] = True
    visibility[2 + 2 * idx + 1] = True
    steps.append(
        dict(
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Healthy WM (MWF = {HEALTHY_MWF}%) vs. Demyelinated Tissue (MWF = {d_mwf}%)"}
            ],
            label=f"{d_mwf}%"
        )
    )

fig.update_layout(
    title=dict(
        text=f"Healthy WM (MWF = {HEALTHY_MWF}%) vs. Demyelinated Tissue (MWF = {initial_dem}%)",
        y=0.96, x=0.5, xanchor="center"
    ),
    template="plotly_white",
    height=560,
    margin=dict(t=90, b=140, l=60, r=40),
    sliders=[
        dict(
            active=demyelinated_values.index(initial_dem),
            currentvalue={"prefix": "Demyelinated tissue MWF: ", "font": {"size": 13}},
            y=-0.18,
            yanchor="top",
            pad={"t": 20, "b": 10},
            steps=steps
        )
    ]
)

fig.update_xaxes(title_text="Echo time TE (ms)", row=1, col=1)
fig.update_xaxes(title_text="T2 relaxation time (ms)", row=1, col=2)
fig.update_yaxes(title_text="Normalized signal S(TE)", range=[0, 1.05], row=1, col=1)
fig.update_yaxes(title_text="Relative amplitude (a.u.)", row=1, col=2)

fig.show()